# 01 Проверка устройства AMD ROCm и окружения

        Цель этого раздела — сначала показать, что устройство, окружение, кэш и цепочка доступа работают. Многие сбои обучения выглядят как проблема модели, а на деле ROCm не видит GPU, кэш лежит не на том диске, сеть не даёт загрузить веса или унифицированную память занимает другой процесс.

        В Notebook в основном делаются три вещи: проверка оборудования и PyTorch, планирование каталогов для больших файлов и составление таблицы ресурсов устройства.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: видит ли ROCm и PyTorch GPU


In [ ]:
def run_cmd(cmd):
    print("$", " ".join(cmd))
    if shutil.which(cmd[0]) is None:
        print(f"Команда не найдена: {cmd[0]}")
        return
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)


run_cmd(["rocm-smi", "--showuse", "--showtemp", "--showmemuse"])

try:
    import torch
    print("torch =", torch.__version__)
    print("torch.cuda.is_available() =", torch.cuda.is_available())
    print("torch.cuda.device_count() =", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("device =", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Ошибка проверки PyTorch:", repr(exc))


Если `rocm-smi` работает, а `torch.cuda.is_available()` возвращает `True`, базовая цепочка ROCm и PyTorch работает. Если первое в порядке, а второе — `False`, сначала проверьте, что установлена сборка PyTorch для ROCm.


## Checkpoint 2: унифицированная память, видеопамять и каталоги на диске


In [ ]:
paths = [
    ("Исходный код проекта", PROJECT_ROOT),
    ("Каталог данных", DATA_ROOT),
    ("Модели и checkpoint", MODEL_ROOT),
    ("Выход Notebook", OUTPUT_ROOT),
    ("Hugging Face cache", Path(os.environ.get("HF_HOME", "/path/to/cache/huggingface"))),
]
md_table(["Элемент", "Рекомендуемый путь", "Существует"], [(name, f"`{path}`", path.exists()) for name, path in paths])


Кэш больших моделей, наборы данных, checkpoint и пакетные видео могут занимать очень много места. В отчёте об эксперименте достаточно описать планировку каталогов и тип диска, эти файлы не нужно класть в каталог учебника.


## Checkpoint 3: шаблон таблицы ресурсов устройства


In [ ]:
rows = [
    ("Модель устройства", "AMD Ryzen AI MAX+ / Radeon GPU"),
    ("Версия системы", "Ubuntu / WSL / другое"),
    ("Версия ROCm", "укажите версию из rocm-smi или пакета"),
    ("Версия PyTorch", "укажите torch.__version__"),
    ("Температура в простое", "например 30-45 C"),
    ("Температура при обучении", "например 75-85 C"),
    ("VRAM при обучении ACT", "укажите значение из rocm-smi"),
    ("VRAM при обучении SmolVLA", "укажите значение из rocm-smi"),
    ("Статус smoke pi0", "не запускался / пройден / причина ошибки"),
]
md_table(["Элемент", "Запись"], rows)


После этого раздела должно быть понятно, готово ли это устройство AMD к дальнейшему обучению и оценке.
